# Half-parameter R-GCN - is it the message passing or the 1.01M parameters?

`ablation_encoder_swap.ipynb` established two points on the capacity curve at a **fixed** graph:

| | encoder params | indication AUPRC |
|---|---|---|
| `rgcn` (library default) | 1,010,000 | 0.8318 |
| `light_gnn` (one shared `Linear`) | 20,200 (2.0%) | 0.6052 |

A 50x cut costs -0.227 indication AUPRC, and since `light_gnn` and `mlp` have **identical** parameter
counts and differ only by the `scatter` over `edge_index`, we already know that message passing *per se*
is nearly free (null for contraindication and micro). So the drop from `rgcn` to `light_gnn` is a
**capacity** effect, not a message-passing one.

What we do not know is the **shape** of that curve between 2% and 100%. This notebook adds the midpoint:
two encoders with **half** the parameters of the original, both still doing full relational message
passing.

| arm | encoder | relational weights | encoder params | % of `rgcn` |
|---|---|---|---|---|
| `rgcn` | `HeteroRGCNLayer` x2 (library default) | 50 independent `Linear(100,100)` | 1,010,000 | 100.00% |
| `rgcn_half_lowrank` | `LowRankRelationLayer` x2 | 50 independent, each **rank 25** | 510,000 | 50.50% |
| `rgcn_half_basis` | `BasisRelationLayer` x2 | full rank, **25 shared bases** | 512,500 | 50.74% |

Both land within 0.8% of exactly half (505,000); the exact integers are forced by the shapes and are
asserted at run time, not assumed.

### Why two different ways of halving

They remove *different* things, and that is the point:

- **`rgcn_half_lowrank`** keeps every relation its own private weight matrix and limits each one to
  rank 25. Relational specificity is fully intact; the expressiveness of each relation's transform is
  halved.
- **`rgcn_half_basis`** is the canonical R-GCN basis decomposition (Schlichtkrull et al. 2018):
  `W_r = sum_b a_rb V_b`, with 25 full-rank bases shared across all 50 relations plus a per-relation
  coefficient vector. Each relation keeps a full-rank transform; what it loses is **independence** from
  the other relations.

This matters because `light_gnn` is exactly `rgcn_half_basis` with **one** basis. The basis arm is
therefore a genuine interpolation point on the `rgcn` -> `light_gnn` line, and the pair of arms
separates two readings of the -0.227 that are otherwise confounded:

- if **basis** holds up and **lowrank** does not, what the R-GCN needs is per-relation *independence*;
- if **lowrank** holds up and **basis** does not, it needs per-relation *rank*;
- if both hold up, the 1.01M parameters are simply over-provisioned and the curve is saturating -
  the single most useful outcome for the comment paper, because it means `rgcn` -> `light_gnn` is a
  cliff somewhere below 50% rather than a smooth slope.

### Why not just narrow the hidden layer

The obvious way to halve an encoder is to shrink `n_hid`, but it does not work here and the reason is
worth recording. `HeteroRGCN` passes `hidden_size` to `DistMultPredictor` as `n_hid`, which sizes the
`rarity` gate as `nn.Linear(n_hid * 2, 1)`, while the node embeddings it concatenates are `out_size`
wide and `w_rels` is `(n_etypes, out_size)`. The library only works because `n_hid == n_out == 100`.
Setting `n_hid = 50` alone is a shape error; setting `n_hid = n_out = 50` changes the **decoder and the
gate** as well, so it would no longer be an encoder-only swap - and it lands at 380,000 params (37.6%),
not half.

Factorising inside the layer keeps the interface at `100 -> 100 -> 100`, so `w_rels`,
`DistMultPredictor`, the `rarity` gate, pretraining and fine-tuning are **bit-for-bit the library's**.
Only the encoder's internal parameterisation changes.

### Initialisation is scale-matched on purpose

A naive factorisation initialises worse than what it replaces: the product of two default-initialised
`nn.Linear` weights has far smaller variance than one, so a "half-size" arm could lose simply because it
started in a worse place. Both layers below therefore initialise their factors with a uniform bound
chosen so that the **effective** weight `W_r` has the same per-entry variance as
`nn.Linear(100, 100)`'s default. The arithmetic is in `_init_factor_` and is unit-tested in section 3.
Without this the experiment would measure initialisation, not capacity.

---
## 1 - Environment

In [ ]:
import os, sys, json, time, gc, random, shutil, inspect, platform, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')


def _find_repo_root(start=None):
    p = Path(start or os.getcwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / 'pyg_implementation' / 'txgnn').is_dir():
            return cand
    raise RuntimeError('Could not find pyg_implementation/txgnn above ' + str(p))


REPO_ROOT = _find_repo_root()
PYG_ROOT  = REPO_ROOT / 'pyg_implementation'

# Put the PyG port FIRST on sys.path so `import txgnn` never resolves to dgl_implementation/.
sys.path.insert(0, str(PYG_ROOT))

import torch
import torch.nn as nn
from torch_scatter import scatter

import txgnn
from txgnn import TxData, TxGNN
from txgnn.model import HeteroRGCN, HeteroRGCNLayer
from txgnn.utils import evaluate_fb, create_pyg_graph, get_n_params

assert 'pyg_implementation' in txgnn.__file__, (
    'Wrong txgnn package on sys.path (%s). Restart the kernel.' % txgnn.__file__)

# The Gumbel-gate switch was removed from the library in commit 2d064cc. Nothing here passes it;
# fail loudly if it ever comes back, rather than silently training a different model.
_MI_PARAMS = set(inspect.signature(TxGNN.model_initialize).parameters)
assert 'use_decision_network' not in _MI_PARAMS, (
    'model_initialize accepts use_decision_network again - revisit this notebook.')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


print('repo        :', REPO_ROOT)
print('txgnn       :', txgnn.__file__)
print('torch       :', torch.__version__, '| device:', DEVICE,
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else platform.processor())

---
## 2 - Configuration

`MODEL_CFG` is **byte-identical** to `ablation_encoder_swap.ipynb` and `ablation_stage_a.ipynb`, including
`n_hid = n_inp = n_out = 100`. The halving happens entirely inside the layer classes, so every arm here
is comparable to every arm in those grids without re-running anything.

`SEEDS` is the same 20-seed set `ablation_encoder_swap` used, which means all 20 `rgcn` reference runs
**import for free** from that grid and only the two new arms need GPU time.

In [ ]:
SPLIT   = 'complex_disease'
SEEDS   = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 31, 41, 42, 63, 70, 77, 86, 89, 95, 98]
PROFILE = 'full'

MODEL_CFG = dict(
    n_hid       = 100,
    n_inp       = 100,
    n_out       = 100,
    proto       = True,                  # metric learning stays ON in every arm here
    proto_num   = 3,
    attention   = False,
    sim_measure = 'all_nodes_profile',
    agg_measure = 'rarity',
)

PRETRAIN_CFG = dict(n_epoch=1,    learning_rate=1e-3, batch_size=1024, train_print_per_n=500)
FINETUNE_CFG = dict(n_epoch=1000, learning_rate=5e-4, train_print_per_n=100, valid_per_n=20)

# --- the two halving knobs -----------------------------------------------------------------------
# Both are chosen to land as close to 50% of 1,010,000 as the shapes allow. Section 3 prints the
# exact counts for a range of values so a different target is a one-line change.
RANK     = 25     # rgcn_half_lowrank: per-relation rank  -> 2*50*(100*r + r*100 + 100)
N_BASES  = 25     # rgcn_half_basis:   shared bases       -> 2*(B*100*100 + 50*B + 50*100)

DD_REL     = ['contraindication', 'indication', 'off-label use']
DD_REV_REL = ['rev_' + r for r in DD_REL]
DD_ETYPES  = [('drug', r, 'disease') for r in DD_REL] + [('disease', r, 'drug') for r in DD_REV_REL]

SIG_ETYPES_DEFAULT = ['disease_disease', 'rev_disease_protein']

DATA_FOLDER = REPO_ROOT / 'data'
ABL_ROOT    = REPO_ROOT / 'results' / 'ablation_encoder_halfparam'
SWAP_ROOT   = REPO_ROOT / 'results' / 'ablation_encoder_swap'   # read-only: ref import + capacity curve
ABL_ROOT.mkdir(parents=True, exist_ok=True)

REF_ARM = 'rgcn'          # every delta is measured against this arm

# --- switches ------------------------------------------------------------------------------------
RUN_GRID          = True   # section 6.2 trains; set False to inspect the plan / aggregate only
SAVE_MODELS       = False
IMPORT_REF        = True   # reuse ablation_encoder_swap's rgcn runs as the reference arm
PROBE_ENCODER     = True   # run the graph-sensitivity check after each run
STOP_AFTER_N_RUNS = None   # e.g. 2 for a smoke test

print('split       :', SPLIT)
print('seeds       : %d -> %s' % (len(SEEDS), SEEDS))
print('results     :', ABL_ROOT.relative_to(REPO_ROOT))
print('rank / bases: %d / %d' % (RANK, N_BASES))

---
## 3 - The two half-size encoders

Both expose exactly `HeteroRGCNLayer`'s interface - `forward(G, feat_dict) -> {ntype: tensor}` - and
reproduce its aggregation to the letter: `scatter`-mean per edge type, sum across edge types, zeros for
node types with no incoming edges, source-side transform applied to the whole node type. The only
difference from the library layer is how `W_r` is stored.

Weights are kept as **stacked tensors** with a name -> index map rather than a `ParameterDict`, which
sidesteps key-name restrictions on relation names like `off-label use` and avoids materialising 50
modules per layer.

Not supported by either: `graphmask_forward` / `add_graphmask_parameter`. The GraphMask explainer would
raise on these arms; it is not used here.

In [ ]:
def _init_factor_(t, fan_in, n_terms):
    '''Uniform-init `t` so the *product-sum* it participates in matches nn.Linear's default scale.

    nn.Linear(fan_in, out) initialises its weight U(-k, k) with k = 1/sqrt(fan_in), so each entry has
    variance k^2/3 = 1/(3*fan_in). Here the effective weight is a sum of `n_terms` products of two
    independently-initialised factors, each U(-c, c) with variance c^2/3, giving

        Var(W_eff) = n_terms * (c^2/3)^2   ->   set equal to 1/(3*fan_in)
        =>  c = (3 / (fan_in * n_terms)) ** 0.25

    Returns c so the caller can log it.
    '''
    c = (3.0 / (fan_in * n_terms)) ** 0.25
    nn.init.uniform_(t, -c, c)
    return c


def _init_bias_(t, fan_in):
    '''nn.Linear's default bias init, applied to our stacked bias tensor.'''
    k = 1.0 / (fan_in ** 0.5)
    nn.init.uniform_(t, -k, k)
    return k


class _StackedRelationLayer(nn.Module):
    '''Shared plumbing: name->index map and HeteroRGCNLayer's exact aggregation.'''

    def __init__(self, in_size, out_size, etypes):
        super().__init__()
        names = list(dict.fromkeys(etypes))          # de-dup, order preserved
        assert len(names) == len(etypes), (
            'relation names are not unique (%d names, %d edge types) - the library would silently '
            'share weights here, so the parameter arithmetic below would be wrong.'
            % (len(names), len(etypes)))
        self.etype_names = names
        self.idx = {n: i for i, n in enumerate(names)}
        self.in_size, self.out_size = in_size, out_size

    def transform(self, etype, x):
        raise NotImplementedError

    def forward(self, G, feat_dict):
        out = {nt: None for nt in G.node_types}

        for srctype, etype, dsttype in G.edge_types:
            ei = G[(srctype, etype, dsttype)].edge_index
            if ei.shape[1] == 0:
                continue
            Wh = self.transform(etype, feat_dict[srctype])
            msg = Wh[ei[0]]
            # scatter mean per etype, then sum across etypes - identical to HeteroRGCNLayer
            agg = scatter(msg, ei[1], dim=0, reduce='mean', dim_size=G[dsttype].num_nodes)
            out[dsttype] = agg if out[dsttype] is None else out[dsttype] + agg

        device = next(iter(feat_dict.values())).device
        for nt in G.node_types:
            if out[nt] is None:
                out[nt] = torch.zeros(G[nt].num_nodes, self.out_size, device=device)
        return out


class LowRankRelationLayer(_StackedRelationLayer):
    '''W_r = A_r @ B_r with A_r: (in, rank), B_r: (rank, out). One private factorisation per relation.

    Per relation: in*rank + rank*out + out parameters.
    '''

    def __init__(self, in_size, out_size, etypes, rank):
        super().__init__(in_size, out_size, etypes)
        n = len(self.etype_names)
        self.rank = rank
        self.A    = nn.Parameter(torch.empty(n, in_size, rank))
        self.B    = nn.Parameter(torch.empty(n, rank, out_size))
        self.bias = nn.Parameter(torch.empty(n, out_size))
        self.init_scale = _init_factor_(self.A, in_size, rank)
        _init_factor_(self.B, in_size, rank)
        _init_bias_(self.bias, in_size)

    def transform(self, etype, x):
        i = self.idx[etype]
        return (x @ self.A[i]) @ self.B[i] + self.bias[i]   # (N,in)@(in,r)@(r,out), never materialises W


class BasisRelationLayer(_StackedRelationLayer):
    '''W_r = sum_b a_rb V_b - the canonical R-GCN basis decomposition (Schlichtkrull et al. 2018).

    Full-rank per-relation transforms, but all relations share `n_bases` basis matrices.
    n_bases = 1 is `SharedRelationLayer` (the `light_gnn` arm) up to the per-relation scalar and bias.

    Total: n_bases*in*out + n_etypes*n_bases + n_etypes*out parameters.
    '''

    def __init__(self, in_size, out_size, etypes, n_bases):
        super().__init__(in_size, out_size, etypes)
        n = len(self.etype_names)
        self.n_bases = n_bases
        self.V       = nn.Parameter(torch.empty(n_bases, in_size, out_size))
        self.coeff   = nn.Parameter(torch.empty(n, n_bases))
        self.bias    = nn.Parameter(torch.empty(n, out_size))
        self.init_scale = _init_factor_(self.V, in_size, n_bases)
        _init_factor_(self.coeff, in_size, n_bases)
        _init_bias_(self.bias, in_size)

    def transform(self, etype, x):
        i = self.idx[etype]
        W = torch.einsum('b,bio->io', self.coeff[i], self.V)   # (100,100), cheap
        return x @ W + self.bias[i]


# --- exact parameter arithmetic -------------------------------------------------------------------
LAYER_SPECS = {
    'rgcn':              dict(cls=HeteroRGCNLayer,        build=lambda i, o, e: HeteroRGCNLayer(i, o, e)),
    'rgcn_half_lowrank': dict(cls=LowRankRelationLayer,   build=lambda i, o, e: LowRankRelationLayer(i, o, e, RANK)),
    'rgcn_half_basis':   dict(cls=BasisRelationLayer,     build=lambda i, o, e: BasisRelationLayer(i, o, e, N_BASES)),
}


def expected_encoder_params(kind, n_etypes, width=100, rank=None, n_bases=None):
    '''Closed form for a 2-layer width->width encoder. Asserted against the real model in run_one.'''
    r = RANK if rank is None else rank
    b = N_BASES if n_bases is None else n_bases
    if kind == 'rgcn':
        return 2 * n_etypes * (width * width + width)
    if kind == 'rgcn_half_lowrank':
        return 2 * n_etypes * (width * r + r * width + width)
    if kind == 'rgcn_half_basis':
        return 2 * (b * width * width + n_etypes * b + n_etypes * width)
    raise ValueError('unknown encoder %r' % kind)


_N_ET = 50          # the full KG; asserted per run against the graph actually built
_FULL = expected_encoder_params('rgcn', _N_ET)
print('rgcn encoder at %d edge types: %s params\n' % (_N_ET, format(_FULL, ',')))

print('rgcn_half_lowrank - pick RANK:')
for r in range(20, 31):
    p = expected_encoder_params('rgcn_half_lowrank', _N_ET, rank=r)
    print('   rank %2d -> %9s  (%6.2f%%)%s' % (r, format(p, ','), 100 * p / _FULL,
                                               '   <- RANK' if r == RANK else ''))
print('\nrgcn_half_basis - pick N_BASES:')
for b in range(20, 31):
    p = expected_encoder_params('rgcn_half_basis', _N_ET, n_bases=b)
    print('   bases %2d -> %9s  (%6.2f%%)%s' % (b, format(p, ','), 100 * p / _FULL,
                                                '   <- N_BASES' if b == N_BASES else ''))

### 3.1 - Unit tests

Before spending GPU hours: the parameter counts must match the closed form exactly, the
initialisation must land on `nn.Linear`'s scale, and the aggregation must be numerically identical to
`HeteroRGCNLayer`'s when the weights are forced to agree. The last test is the important one - it is
what licenses calling these arms "the same encoder with fewer parameters".

In [ ]:
from torch_geometric.data import HeteroData

_ets = ['r%d' % i for i in range(_N_ET)]

# (1) parameter counts match the closed form, exactly
for kind, spec in LAYER_SPECS.items():
    l1 = spec['build'](100, 100, _ets)
    l2 = spec['build'](100, 100, _ets)
    got  = sum(p.numel() for p in l1.parameters()) + sum(p.numel() for p in l2.parameters())
    want = expected_encoder_params(kind, _N_ET)
    assert got == want, '%s: %d != %d' % (kind, got, want)
    print('[params] %-18s %9s  (%6.2f%% of rgcn)  OK' % (kind, format(got, ','), 100 * got / _FULL))

# (2) effective weight variance matches nn.Linear's default init
_ref_var = nn.Linear(100, 100).weight.var().item()
for kind in ['rgcn_half_lowrank', 'rgcn_half_basis']:
    l = LAYER_SPECS[kind]['build'](100, 100, _ets)
    with torch.no_grad():
        if kind == 'rgcn_half_lowrank':
            W = torch.stack([l.A[i] @ l.B[i] for i in range(len(_ets))])
        else:
            W = torch.einsum('nb,bio->nio', l.coeff, l.V)
    ratio = W.var().item() / _ref_var
    assert 0.5 < ratio < 2.0, '%s init variance ratio %.3f is off' % (kind, ratio)
    print('[init]   %-18s Var(W_eff)/Var(Linear) = %.3f  (c = %.4f)  OK' % (kind, ratio, l.init_scale))

# (3) aggregation is identical to HeteroRGCNLayer when the effective weights are made to agree
_g = HeteroData()
_g['a'].num_nodes, _g['b'].num_nodes = 7, 5
_g['a', 'r0', 'b'].edge_index = torch.tensor([[0, 1, 1, 3, 6], [0, 0, 2, 4, 4]])
_g['b', 'r1', 'a'].edge_index = torch.tensor([[0, 2, 4], [1, 1, 5]])
_g['a', 'r2', 'b'].edge_index = torch.zeros((2, 0), dtype=torch.long)       # empty: must be skipped
_feat = {'a': torch.randn(7, 100), 'b': torch.randn(5, 100)}
_names = [et[1] for et in _g.edge_types]

for kind in ['rgcn_half_lowrank', 'rgcn_half_basis']:
    lay = LAYER_SPECS[kind]['build'](100, 100, _names)
    ref = HeteroRGCNLayer(100, 100, _names)
    with torch.no_grad():
        for nm in _names:                      # force the stock layer to use OUR effective weights
            i = lay.idx[nm]
            W = (lay.A[i] @ lay.B[i]) if kind == 'rgcn_half_lowrank' else \
                torch.einsum('b,bio->io', lay.coeff[i], lay.V)
            ref.weight[nm].weight.copy_(W.T)   # nn.Linear stores (out, in)
            ref.weight[nm].bias.copy_(lay.bias[i])
        o_new, o_ref = lay(_g, _feat), ref(_g, _feat)
    for nt in _g.node_types:
        d = (o_new[nt] - o_ref[nt]).abs().max().item()
        assert d < 1e-4, '%s aggregation differs on %s by %g' % (kind, nt, d)
    # the isolated node type must be exactly zero-filled, as in the library
    assert o_new['a'][[0, 2, 3, 4, 6]].abs().sum().item() >= 0
    print('[agg]    %-18s max |delta| vs HeteroRGCNLayer = %.2e  OK' % (kind, d))

del _g, _feat, _ets
print('\nall unit tests passed')

---
## 4 - The arms

`rgcn` is the reference and comes in free from `ablation_encoder_swap`. Only the two half-size arms
need training.

In [ ]:
ARMS = [
    {'arm': 'rgcn',              'encoder': 'rgcn',              'overrides': {}},
    {'arm': 'rgcn_half_lowrank', 'encoder': 'rgcn_half_lowrank', 'overrides': {}},
    {'arm': 'rgcn_half_basis',   'encoder': 'rgcn_half_basis',   'overrides': {}},
]

ARMS_BY_NAME = {a['arm']: a for a in ARMS}
assert len(ARMS_BY_NAME) == len(ARMS), 'duplicate arm name'
assert REF_ARM in ARMS_BY_NAME, 'REF_ARM %r is not an arm' % REF_ARM
assert all(a['encoder'] in LAYER_SPECS for a in ARMS), 'unknown encoder in ARMS'

print('%d arms x %d seeds = %d runs' % (len(ARMS), len(SEEDS), len(ARMS) * len(SEEDS)))
print('(%d of those import from ablation_encoder_swap, so %d need GPU time)\n'
      % (len(SEEDS), (len(ARMS) - 1) * len(SEEDS)))
for a in ARMS:
    p = expected_encoder_params(a['encoder'], _N_ET)
    print('  %-18s encoder=%-18s params %9s  (%6.2f%%)'
          % (a['arm'], a['encoder'], format(p, ','), 100 * p / _FULL))

---
## 5 - Mechanics and plan

**`RunData`** rebuilds `G` from the untouched `df_train` for every run, so no run inherits a graph
another run moved to the GPU or re-embedded. Nothing is filtered in this notebook - every arm sees all
50 edge types and all 6,735,310 training edges.

**`install_encoder`** rebinds `HeteroRGCN` in the `txgnn.TxGNN` module namespace. `model_initialize`
resolves the name from that namespace (`TxGNN.py` does `from .model import *`), so this swaps the
encoder without touching the library. `txgnn.TxGNN` the *attribute* is the class, so the module is
reached through `sys.modules`.

**`encoder_reads_graph`** re-runs the encoder with every `edge_index` emptied and reports the largest
change in node embeddings. Both arms here do message passing, so both must come back **non-zero** -
this is the guard against a factorisation bug that accidentally drops the aggregation. It runs after
metrics are computed and restores `edge_index` in a `finally` block.

**Importing the reference arm.** `rgcn` is configuration-identical to `ablation_encoder_swap/rgcn`
(itself partly imported from Stage A's `baseline`). With `IMPORT_REF = True` each seed is copied after
checking the recorded `split`, `seed`, `model_config` (the eight keys this notebook sets), `pretrain`,
`finetune`, `dropped_edges_total == 0`, `signature_disease_etypes` and `proto`. Any mismatch falls
through to a normal run; imported records carry `imported_from`.

In [ ]:
class RunData:
    '''Minimal TxData stand-in. Graph rebuilt per run; nothing filtered in this notebook.'''

    def __init__(self, base, data_folder, split):
        self.df       = base['df']
        self.df_train = base['df_train']
        self.df_valid = base['df_valid']
        self.df_test  = base['df_test']
        self.G = create_pyg_graph(self.df_train, self.df)
        self.data_folder = str(data_folder)
        self.split = split
        self.disease_eval_idx = None
        self.no_kg = False


_SEED_CACHE = {}


def get_seed_data(seed):
    '''Load (and cache) the split frames for one seed. Holds one seed at a time - `df` is huge.'''
    if seed in _SEED_CACHE:
        return _SEED_CACHE[seed]
    _SEED_CACHE.clear()
    gc.collect()
    print('[data] loading split seed %d ...' % seed)
    t0 = time.time()
    set_all_seeds(seed)
    d = TxData(data_folder_path=str(DATA_FOLDER))
    d.prepare_split(split=SPLIT, seed=seed, no_kg=False)
    base = {'df': d.df, 'df_train': d.df_train, 'df_valid': d.df_valid, 'df_test': d.df_test}
    del d
    gc.collect()
    _SEED_CACHE[seed] = base
    print('[data] seed %d ready in %.1f s | train edges %d'
          % (seed, time.time() - t0, len(base['df_train'])))
    return base


def run_dir(arm, seed):
    return ABL_ROOT / ('%s_seed%d' % (arm, seed))


def is_done(arm, seed):
    d = run_dir(arm, seed)
    return (d / 'edge_level_test_metrics.csv').exists() and (d / 'run_meta.json').exists()


def encoder_params(model):
    return (sum(p.numel() for p in model.layer1.parameters())
            + sum(p.numel() for p in model.layer2.parameters()))


def encoder_reads_graph(model, G):
    '''Max |delta| in node embeddings when every edge_index is emptied. 0.0 => no message passing.'''
    model.eval()
    saved = {et: G[et].edge_index for et in G.edge_types}
    empty = torch.zeros((2, 0), dtype=torch.long, device=saved[next(iter(saved))].device)
    try:
        with torch.no_grad():
            h_full = model(G, G, return_h=True)
            for et in G.edge_types:
                G[et].edge_index = empty
            h_none = model(G, G, return_h=True)
        return float(max((h_full[nt] - h_none[nt]).abs().max().item() for nt in h_full))
    finally:
        for et, ei in saved.items():
            G[et].edge_index = ei


# --- installing an encoder ------------------------------------------------------------------------
_TXGNN_MODULE    = sys.modules['txgnn.TxGNN']
_ORIG_HETERORGCN = HeteroRGCN


def make_encoder_class(kind):
    build = LAYER_SPECS[kind]['build']

    class SwappedHeteroRGCN(_ORIG_HETERORGCN):
        '''HeteroRGCN with layer1/layer2 replaced. Everything else is the library's.'''

        def __init__(self, G, **kw):
            # super() builds the stock R-GCN layers and we then discard them. Cheap (<1 s) but it does
            # consume RNG draws, so the parameter init stream differs from the rgcn arm. That is fine -
            # each arm is seeded independently and is deterministic within itself.
            super().__init__(G, **kw)
            etypes = [et[1] for et in G.edge_types]
            self.layer1 = build(kw['in_size'],     kw['hidden_size'], etypes)
            self.layer2 = build(kw['hidden_size'], kw['out_size'],    etypes)
            self.encoder_kind = kind

    SwappedHeteroRGCN.__name__ = 'HeteroRGCN_' + kind
    return SwappedHeteroRGCN


def install_encoder(kind):
    '''Point model_initialize at the encoder for `kind`. Returns the class it installed.'''
    if kind not in LAYER_SPECS:
        raise ValueError('unknown encoder %r' % kind)
    cls = _ORIG_HETERORGCN if kind == 'rgcn' else make_encoder_class(kind)
    _TXGNN_MODULE.HeteroRGCN = cls
    return cls


install_encoder('rgcn')     # leave the library default in place until a run asks otherwise
print('encoders available:', sorted(LAYER_SPECS))
print('txgnn.TxGNN.HeteroRGCN ->', _TXGNN_MODULE.HeteroRGCN.__name__)

In [ ]:
# --- import pass, then the plan -------------------------------------------------------------------
MODEL_KEYS_CHECKED = sorted(MODEL_CFG)


def import_matching_run(src, dst, expect, extra=None):
    '''Copy a finished run from another grid iff its metadata matches `expect`. Returns (ok, why).'''
    sm, se = src / 'run_meta.json', src / 'edge_level_test_metrics.csv'
    if not (sm.exists() and se.exists()):
        return False, 'no run at %s' % src.name
    meta = json.loads(sm.read_text())
    for k, want in expect.items():
        got = meta.get(k, '<missing>')
        if isinstance(want, dict):
            if not isinstance(got, dict):
                return False, '%s is not a dict' % k
            bad = {kk: (got.get(kk, '<missing>'), vv) for kk, vv in want.items()
                   if got.get(kk, '<missing>') != vv}
            if bad:
                return False, '%s mismatch %s' % (k, bad)
        elif got != want:
            return False, '%s mismatch: %r != %r' % (k, got, want)

    dst.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(se, dst / 'edge_level_test_metrics.csv')
    meta.update(extra or {})
    meta['imported_from'] = str(src.relative_to(REPO_ROOT))
    meta['run_key'] = dst.name
    meta['arm'] = dst.name.rsplit('_seed', 1)[0]
    with open(dst / 'run_meta.json', 'w') as f:
        json.dump(meta, f, indent=2, default=str)
    return True, 'imported from %s' % src.relative_to(REPO_ROOT)


def try_import_ref(seed):
    if not IMPORT_REF:
        return False, 'disabled'
    return import_matching_run(
        SWAP_ROOT / ('rgcn_seed%d' % seed),
        run_dir(REF_ARM, seed),
        expect={'split': SPLIT, 'seed': seed,
                'dropped_edges_total': 0,
                'proto': True,
                'signature_disease_etypes': SIG_ETYPES_DEFAULT,
                'model_config': {k: MODEL_CFG[k] for k in MODEL_KEYS_CHECKED},
                'pretrain': PRETRAIN_CFG, 'finetune': FINETUNE_CFG},
        extra={'encoder': 'rgcn', 'layer_class': 'HeteroRGCNLayer',
               'message_passing': True, 'experiment': 'encoder_halfparam',
               'encoder_params': expected_encoder_params('rgcn', _N_ET),
               'rank': None, 'n_bases': None},
    )


if IMPORT_REF:
    n_ok = 0
    for s in SEEDS:
        if is_done(REF_ARM, s):
            n_ok += 1
            continue
        ok, why = try_import_ref(s)
        n_ok += bool(ok)
        print('[import] %s_seed%d: %s' % (REF_ARM, s, why if ok else 'will train (%s)' % why))
    print('[import] %d/%d reference runs available\n' % (n_ok, len(SEEDS)))

# --- timing estimate straight off disk ------------------------------------------------------------
BASE_PRETRAIN_S, BASE_FINETUNE_S = 1155.7, 316.5
_times = []
for _p in sorted(SWAP_ROOT.glob('rgcn_seed*/run_meta.json')):
    _r = json.loads(_p.read_text())
    if _r.get('pretrain_seconds') and _r.get('finetune_seconds'):
        _times.append((_r['pretrain_seconds'], _r['finetune_seconds']))
if _times:
    BASE_PRETRAIN_S = float(np.mean([t[0] for t in _times]))
    BASE_FINETUNE_S = float(np.mean([t[1] for t in _times]))
EST_MIN = (BASE_PRETRAIN_S + BASE_FINETUNE_S) / 60

rows = []
for a in ARMS:
    done = [s for s in SEEDS if is_done(a['arm'], s)]
    p = expected_encoder_params(a['encoder'], _N_ET)
    rows.append({'arm': a['arm'], 'encoder': a['encoder'],
                 'encoder params': '%s (%.2f%%)' % (format(p, ','), 100 * p / _FULL),
                 'done': len(done), 'todo': len(SEEDS) - len(done)})
display(pd.DataFrame(rows).style.hide(axis='index'))

RUN_KEYS = [(a['arm'], s) for s in SEEDS for a in ARMS]      # seed-major, matches the run order
pending  = [k for k in RUN_KEYS if not is_done(*k)]
print('\nrun_keys (%d total, %d done, %d pending):'
      % (len(RUN_KEYS), len(RUN_KEYS) - len(pending), len(pending)))
for arm, seed in RUN_KEYS:
    print('  [%s] %s_seed%d' % ('x' if is_done(arm, seed) else ' ', arm, seed))
print('\nestimated remaining: %.1f GPU-hours at ~%.1f min/run (pretrain %.0f s + finetune %.0f s'
      ' averaged over %d rgcn runs on disk)'
      % (len(pending) * EST_MIN / 60, EST_MIN, BASE_PRETRAIN_S, BASE_FINETUNE_S, len(_times)))
print('order: seed-major - after seed %d finishes, every arm has n=1 and can already be read.' % SEEDS[0])

---
## 6 - The grid

Each finished run writes `edge_level_test_metrics.csv` and `run_meta.json` into
`results/ablation_encoder_halfparam/<run_key>/`. The metadata records the encoder kind, the layer class
actually installed, the exact encoder and total parameter counts, `rank` / `n_bases`, the
`encoder_reads_graph` probe and the timings.

`run_meta.json` is written **last**, and `is_done` requires both files, so an interrupted run is left
pending and is redone from scratch on the next pass. The notebook is safe to re-run from cell 1 at any
point: finished `(arm, seed)` pairs are skipped.

Three hard guards fire before any training happens:

1. the installed class is the one `install_encoder` returned, and `layer1` is the expected layer class;
2. `encoder_params(model)` equals `expected_encoder_params(...)` **exactly**, computed from the number
   of edge types in the graph actually built - so a "half" arm that is not half fails loudly;
3. the KG is intact (both signature relations present, 50 edge types).

In [ ]:
def run_one(arm_name, seed, verbose=True):
    '''Train one (arm, seed) and write its results. Returns the edge-level DataFrame.'''
    arm  = ARMS_BY_NAME[arm_name]
    rdir = run_dir(arm_name, seed)
    el_csv, meta_json = rdir / 'edge_level_test_metrics.csv', rdir / 'run_meta.json'

    if el_csv.exists() and meta_json.exists():
        if verbose:
            print('[skip] %s_seed%d -> cached' % (arm_name, seed))
        return pd.read_csv(el_csv)

    if arm_name == REF_ARM:
        ok, why = try_import_ref(seed)
        if ok:
            print('[skip] %s_seed%d -> %s' % (arm_name, seed, why))
            return pd.read_csv(el_csv)

    cfg  = dict(MODEL_CFG, **arm['overrides'])
    kind = arm['encoder']

    print('\n' + '#' * 78)
    print('# %s   seed %d   encoder=%s   rank=%s   bases=%s'
          % (arm_name, seed, kind,
             RANK if kind == 'rgcn_half_lowrank' else '-',
             N_BASES if kind == 'rgcn_half_basis' else '-'))
    print('#' * 78)

    base = get_seed_data(seed)

    set_all_seeds(seed)
    d = RunData(base, DATA_FOLDER, SPLIT)
    etypes_after = sorted('%s|%s|%s' % et for et in d.G.edge_types)
    n_et = len(d.G.edge_types)

    # the KG must be intact in every arm of this notebook
    rel_names = {et[1] for et in d.G.edge_types}
    missing = [r for r in SIG_ETYPES_DEFAULT if r not in rel_names]
    assert not missing, 'KG is not intact, missing from G: %s' % missing
    print('[graph] train edges %d | edge types %d (KG intact in every arm)' % (len(d.df_train), n_et))

    installed = install_encoder(kind)

    set_all_seeds(seed)
    m = TxGNN(data=d, weight_bias_track=False, exp_name='%s_seed%d' % (arm_name, seed), device=DEVICE)
    m.model_initialize(**cfg)

    # ---- guard 1: the swap took ----
    assert type(m.model) is installed, 'encoder swap did not take: %s' % type(m.model).__name__
    layer_cls = type(m.model.layer1).__name__
    assert layer_cls == LAYER_SPECS[kind]['cls'].__name__, \
        'layer1 is %s, expected %s' % (layer_cls, LAYER_SPECS[kind]['cls'].__name__)

    # ---- guard 2: the arm really has the parameter count it claims ----
    enc_p, tot_p = encoder_params(m.model), get_n_params(m.model)
    want_p = expected_encoder_params(kind, n_et)
    assert enc_p == want_p, 'encoder params %d != expected %d for %s at %d edge types' \
                            % (enc_p, want_p, kind, n_et)
    full_p = expected_encoder_params('rgcn', n_et)
    print('[model] %s | layer=%s | encoder params %s = %.2f%% of rgcn | total %s'
          % (type(m.model).__name__, layer_cls, format(enc_p, ','), 100 * enc_p / full_p,
             format(tot_p, ',')))

    t0 = time.time()
    set_all_seeds(seed)
    m.pretrain(**PRETRAIN_CFG)
    pre_s = time.time() - t0

    t0 = time.time()
    set_all_seeds(seed)
    m.finetune(**FINETUNE_CFG)
    fin_s = time.time() - t0

    # ---- edge-level test metrics ----
    (auroc_rel, auprc_rel, micro_auroc, micro_auprc, macro_auroc, macro_auprc), test_loss = \
        evaluate_fb(m.best_model, m.g_test_pos, m.g_test_neg, m.G, m.dd_etypes, DEVICE, mode='test')

    el = pd.DataFrame(
        [('%s --%s--> %s' % et, auroc_rel.get(et, np.nan), auprc_rel.get(et, np.nan))
         for et in DD_ETYPES],
        columns=['relation', 'AUROC', 'AUPRC'])
    el.loc[len(el)] = ['MICRO (drug-disease)', micro_auroc, micro_auprc]
    el.loc[len(el)] = ['MACRO (drug-disease)',
                       float(np.nanmean([auroc_rel.get(et, np.nan) for et in DD_ETYPES])),
                       float(np.nanmean([auprc_rel.get(et, np.nan) for et in DD_ETYPES]))]
    el.loc[len(el)] = ['MACRO (all relations)', macro_auroc, macro_auprc]

    rdir.mkdir(parents=True, exist_ok=True)
    el.to_csv(el_csv, index=False)

    # ---- guard 3, after metrics so it cannot perturb training ----
    graph_sensitivity = None
    if PROBE_ENCODER:
        graph_sensitivity = encoder_reads_graph(m.best_model, m.G)
        print('[probe] max |delta h| with all edges removed: %.6g' % graph_sensitivity)
        # every arm here does message passing - a zero would mean the factorisation dropped the scatter
        assert graph_sensitivity > 1e-6, \
            '%s does not read the graph (delta %g) - aggregation is broken' % (kind, graph_sensitivity)

    meta = {
        'run_key': '%s_seed%d' % (arm_name, seed),
        'arm': arm_name, 'seed': seed, 'split': SPLIT, 'profile': PROFILE,
        'experiment': 'encoder_halfparam',
        'encoder': kind,
        'encoder_class': type(m.model).__name__,
        'layer_class': layer_cls,
        'rank':    RANK    if kind == 'rgcn_half_lowrank' else None,
        'n_bases': N_BASES if kind == 'rgcn_half_basis'   else None,
        'message_passing': True,
        'graph_sensitivity': graph_sensitivity,
        'encoder_params': enc_p,
        'encoder_params_pct_of_rgcn': 100.0 * enc_p / full_p,
        'n_params': tot_p,
        'proto': cfg['proto'],
        'signature_disease_etypes': SIG_ETYPES_DEFAULT if cfg['proto'] else [],
        'dropped_edges_total': 0,
        'train_edges': len(d.df_train),
        'n_edge_types_after': n_et,
        'edge_types_after': etypes_after,
        'model_config': m.config,
        'pretrain': PRETRAIN_CFG, 'finetune': FINETUNE_CFG,
        'pretrain_seconds': pre_s, 'finetune_seconds': fin_s,
        'test_bce_loss': float(test_loss),
        'torch': torch.__version__,
        'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
        'finished_at': time.strftime('%Y-%m-%dT%H:%M:%S'),
    }
    # written last - is_done() needs both files, so a crash mid-write leaves the run pending
    with open(meta_json, 'w') as f:
        json.dump(meta, f, indent=2, default=str)

    if SAVE_MODELS:
        mdir = REPO_ROOT / 'saved_models' / 'ablation_encoder_halfparam' / meta['run_key']
        mdir.mkdir(parents=True, exist_ok=True)
        m.save_model(str(mdir))

    ind = auprc_rel.get(('drug', 'indication', 'disease'), np.nan)
    con = auprc_rel.get(('drug', 'contraindication', 'disease'), np.nan)
    print('[done] %s_seed%d | %.1f min | indication AUPRC %.4f | contraindication AUPRC %.4f -> %s'
          % (arm_name, seed, (pre_s + fin_s) / 60, ind, con, rdir.relative_to(REPO_ROOT)))

    # ---- free the GPU and restore the library encoder before the next run ----
    del m, d, auroc_rel, auprc_rel
    install_encoder('rgcn')
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return el

In [ ]:
if RUN_GRID:
    todo = [(a['arm'], s) for s in SEEDS for a in ARMS if not is_done(a['arm'], s)]
    if STOP_AFTER_N_RUNS:
        todo = todo[:STOP_AFTER_N_RUNS]

    print('%d run(s) pending of %d total.' % (len(todo), len(ARMS) * len(SEEDS)))
    grid_t0 = time.time()
    for i, (arm_name, seed) in enumerate(todo, 1):
        print('\n>>> %d/%d  %s_seed%d  (elapsed %.1f h)'
              % (i, len(todo), arm_name, seed, (time.time() - grid_t0) / 3600))
        run_one(arm_name, seed)
    print('\nGrid finished %d run(s) in %.1f h.' % (len(todo), (time.time() - grid_t0) / 3600))
else:
    print('RUN_GRID is False - no training. Set it in section 2.')

---
## 7 - Aggregate

Deltas are paired per seed against `rgcn`, the same statistic every other grid uses.

In [ ]:
def _load_all(root, experiment=None):
    recs = []
    for meta_p in sorted(Path(root).glob('*/run_meta.json')):
        el_p = meta_p.parent / 'edge_level_test_metrics.csv'
        if not el_p.exists():
            continue
        meta = json.loads(meta_p.read_text())
        el = pd.read_csv(el_p).set_index('relation')
        for rel, row in el.iterrows():
            recs.append({'arm': meta['arm'], 'seed': meta['seed'], 'relation': rel,
                         'AUROC': row['AUROC'], 'AUPRC': row['AUPRC'],
                         'encoder': meta.get('encoder'), 'proto': meta.get('proto'),
                         'message_passing': meta.get('message_passing'),
                         'encoder_params': meta.get('encoder_params'),
                         'n_params': meta.get('n_params'),
                         'rank': meta.get('rank'), 'n_bases': meta.get('n_bases'),
                         'grid': experiment or Path(root).name,
                         'imported': 'imported_from' in meta})
    return pd.DataFrame(recs)


def _ci95(x):
    x = np.asarray(x, dtype=float)
    return 1.96 * x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else np.nan


ROW = {'indication': 'drug --indication--> disease',
       'contraindication': 'drug --contraindication--> disease',
       'micro': 'MICRO (drug-disease)'}

long = _load_all(ABL_ROOT)
summary, wide, ref = None, None, None

if long.empty:
    print('No runs on disk yet - run the grid cell in section 6 first.')
else:
    wide = (long[long.relation.isin(ROW.values())]
            .pivot_table(index=['arm', 'seed'], columns='relation', values='AUPRC')
            .rename(columns={v: k for k, v in ROW.items()}))

    _first = lambda s: s.dropna().iloc[0] if s.notna().any() else np.nan
    info = (long.groupby('arm')
                .agg(encoder=('encoder', 'first'), proto=('proto', 'first'),
                     encoder_params=('encoder_params', _first),
                     n_params=('n_params', _first),
                     rank=('rank', _first), n_bases=('n_bases', _first)))

    if REF_ARM in wide.index.get_level_values('arm'):
        ref = wide.xs(REF_ARM, level='arm')
    else:
        print('[warn] no %s runs yet - deltas will be NaN.' % REF_ARM)

    out = []
    for arm, g in wide.groupby(level='arm'):
        g = g.droplevel('arm')
        rec = {'arm': arm, 'n_seeds': len(g),
               'encoder': info.loc[arm, 'encoder'],
               'encoder_params': info.loc[arm, 'encoder_params'],
               'rank': info.loc[arm, 'rank'], 'n_bases': info.loc[arm, 'n_bases']}
        for key in ROW:
            rec['%s_AUPRC' % key] = g[key].mean()
            rec['%s_CI' % key] = _ci95(g[key])
            if arm == REF_ARM:
                rec['%s_delta' % key], rec['%s_delta_CI' % key] = 0.0, np.nan
                rec['%s_n_paired' % key] = len(g)
            elif ref is not None:
                paired = (g[key] - ref[key]).dropna()
                rec['%s_delta' % key] = paired.mean() if len(paired) else np.nan
                rec['%s_delta_CI' % key] = _ci95(paired)
                rec['%s_n_paired' % key] = len(paired)
                rec['%s_hurt' % key] = int((paired < 0).sum())
            else:
                rec['%s_delta' % key], rec['%s_delta_CI' % key] = np.nan, np.nan
        out.append(rec)

    order = [a['arm'] for a in ARMS]
    summary = (pd.DataFrame(out)
                 .assign(_o=lambda d: d.arm.map({a: i for i, a in enumerate(order)}))
                 .sort_values('_o').drop(columns='_o').reset_index(drop=True))
    summary.to_csv(ABL_ROOT / 'encoder_halfparam_summary.csv', index=False)

    cols = ['arm', 'n_seeds', 'encoder_params',
            'indication_AUPRC', 'indication_CI', 'indication_delta', 'indication_delta_CI',
            'indication_hurt',
            'contraindication_AUPRC', 'contraindication_delta', 'contraindication_delta_CI',
            'contraindication_hurt', 'micro_AUPRC', 'micro_delta']
    cols = [c for c in cols if c in summary.columns]
    print('%d arms, %d runs on disk -> %s\n'
          % (len(summary), len(wide), (ABL_ROOT / 'encoder_halfparam_summary.csv').relative_to(REPO_ROOT)))
    fmt = [c for c in cols if c.endswith(('AUPRC', 'CI', 'delta'))]
    display(summary[cols].style.format('{:.4f}', subset=fmt, na_rep='-')
                         .format({'encoder_params': '{:,.0f}'}, na_rep='-').hide(axis='index'))

    short = summary[summary.n_seeds < len(SEEDS)]
    if len(short):
        print('\n[partial] below %d seeds, deltas provisional: %s'
              % (len(SEEDS), ', '.join('%s (n=%d)' % (r.arm, r.n_seeds) for r in short.itertuples())))

### 7.1 - The capacity curve

This is the point of the notebook. It pools these arms with `ablation_encoder_swap`'s so the full curve
is in one table, at a **fixed** graph, all arms with metric learning on:

| encoder params | arm | grid |
|---|---|---|
| 1,010,000 | `rgcn` | encoder_swap |
| 512,500 | `rgcn_half_basis` | here |
| 510,000 | `rgcn_half_lowrank` | here |
| 20,200 | `light_gnn` | encoder_swap |
| 20,200 | `mlp` (no message passing) | encoder_swap |

`mlp` is shown for reference but is **off-curve** - it is the only arm without message passing.

How to read the shape:

- **Saturating** (both half arms within ~0.02 of `rgcn`): the 1.01M parameters are over-provisioned and
  the `rgcn` -> `light_gnn` collapse is a cliff that happens somewhere well below 50%. Strengthens the
  comment paper's capacity argument considerably.
- **Linear** (both half arms roughly halfway to `light_gnn`, about -0.11 indication): capacity buys
  performance smoothly, and "the encoder matters" becomes "encoder size matters", which is a weaker but
  cleaner claim.
- **Split** (one half arm holds, the other does not): the binding constraint is named - per-relation
  rank if `lowrank` fails, per-relation independence if `basis` fails.

In [ ]:
CURVE_IMPORT = ['light_gnn', 'mlp']       # pulled from ablation_encoder_swap; rgcn comes from here

swap_long = _load_all(SWAP_ROOT, 'encoder_swap') if SWAP_ROOT.exists() else pd.DataFrame()
curve = None

if summary is None:
    print('Need finished runs here first.')
else:
    pieces = [long[long.arm.isin([a['arm'] for a in ARMS])]]
    if not swap_long.empty:
        pieces.append(swap_long[swap_long.arm.isin(CURVE_IMPORT) & (swap_long.arm != REF_ARM)])
    pool = pd.concat(pieces, ignore_index=True)

    w = (pool[pool.relation.isin(ROW.values())]
         .pivot_table(index=['arm', 'seed'], columns='relation', values='AUPRC')
         .rename(columns={v: k for k, v in ROW.items()}))
    _first = lambda s: s.dropna().iloc[0] if s.notna().any() else np.nan
    pinfo = pool.groupby('arm').agg(encoder_params=('encoder_params', _first),
                                    message_passing=('message_passing', _first),
                                    grid=('grid', 'first'))
    # encoder_params is missing from the oldest imported records - fill from the closed form
    KNOWN_P = {'rgcn': _FULL, 'light_gnn': 20200, 'mlp': 20200,
               'rgcn_half_lowrank': expected_encoder_params('rgcn_half_lowrank', _N_ET),
               'rgcn_half_basis':   expected_encoder_params('rgcn_half_basis', _N_ET)}
    pref = w.xs(REF_ARM, level='arm') if REF_ARM in w.index.get_level_values('arm') else None

    rows = []
    for arm in w.index.get_level_values('arm').unique():
        g = w.xs(arm, level='arm')
        p = pinfo.loc[arm, 'encoder_params']
        p = KNOWN_P.get(arm, np.nan) if pd.isna(p) else p
        mp = pinfo.loc[arm, 'message_passing']
        rec = {'arm': arm, 'grid': pinfo.loc[arm, 'grid'],
               'message passing': bool(mp) if pd.notna(mp) else None,
               'encoder params': p, '% of rgcn params': 100.0 * p / _FULL,
               'n_seeds': len(g)}
        for key in ['indication', 'contraindication', 'micro']:
            rec['%s AUPRC' % key] = g[key].mean()
            if pref is not None and arm != REF_ARM:
                paired = (g[key] - pref[key]).dropna()
                rec['%s delta' % key] = paired.mean() if len(paired) else np.nan
                rec['%s n_paired' % key] = len(paired)
            elif arm == REF_ARM:
                rec['%s delta' % key] = 0.0
                rec['%s n_paired' % key] = len(g)
        rows.append(rec)

    curve = (pd.DataFrame(rows)
               .sort_values('encoder params', ascending=False)
               .reset_index(drop=True))
    curve.to_csv(ABL_ROOT / 'capacity_curve.csv', index=False)

    show = ['arm', 'grid', 'message passing', 'encoder params', '% of rgcn params', 'n_seeds',
            'indication AUPRC', 'indication delta', 'contraindication AUPRC',
            'contraindication delta', 'micro AUPRC', 'micro delta']
    display(curve[show].style
                .format({'encoder params': '{:,.0f}', '% of rgcn params': '{:.2f}'}, na_rep='-')
                .format('{:.4f}', subset=[c for c in show if c.endswith(('AUPRC', 'delta'))], na_rep='-')
                .hide(axis='index'))
    print('-> %s' % (ABL_ROOT / 'capacity_curve.csv').relative_to(REPO_ROOT))

    # --- verdict on the shape ---------------------------------------------------------------------
    c = curve.set_index('arm')
    halves = [a for a in ['rgcn_half_basis', 'rgcn_half_lowrank'] if a in c.index]
    if halves and 'light_gnn' in c.index:
        lg = c.loc['light_gnn', 'indication delta']
        print('\nindication delta vs rgcn:   light_gnn (2%% of params) = %+.4f' % lg)
        for a in halves:
            d = c.loc[a, 'indication delta']
            frac = d / lg if lg else np.nan
            print('  %-18s (%5.2f%% of params) = %+.4f  -> %.0f%% of the light_gnn drop at 50%% of the'
                  ' parameters' % (a, c.loc[a, '% of rgcn params'], d, 100 * frac))
        worst = min(abs(c.loc[a, 'indication delta']) for a in halves)
        if worst < 0.02:
            shape = 'SATURATING - 1.01M parameters are over-provisioned; the collapse is a cliff below 50%'
        elif all(0.3 <= (c.loc[a, 'indication delta'] / lg) <= 0.7 for a in halves):
            shape = 'LINEAR - performance tracks encoder size smoothly'
        else:
            shape = 'SPLIT - the two halving schemes disagree; see which factor binds'
        print('\nshape: %s' % shape)
    else:
        print('\n(need light_gnn from ablation_encoder_swap plus at least one half arm for the verdict)')

In [ ]:
# Per-seed paired deltas - the sign-consistency check a mean hides.
if summary is not None and ref is not None:
    for key in ['indication', 'contraindication']:
        arms = [a for a in summary.arm if a != REF_ARM]
        tbl = pd.DataFrame({a: (wide.xs(a, level='arm')[key] - ref[key]) for a in arms}).T
        seeds_present = list(tbl.columns)
        tbl.columns = ['seed %d' % s_ for s_ in seeds_present]
        tbl['mean'] = tbl.mean(axis=1)
        tbl['hurt'] = (tbl[['seed %d' % s_ for s_ in seeds_present]] < 0).sum(axis=1)
        print('\npaired delta AUPRC vs %s - %s  (negative = halving hurt)' % (REF_ARM, key))
        display(tbl.style.format('{:.4f}', subset=[c for c in tbl.columns if c != 'hurt'], na_rep='-'))

    # head-to-head: does rank or independence bind? paired on common seeds, rgcn drops out
    if {'rgcn_half_lowrank', 'rgcn_half_basis'} <= set(summary.arm):
        print('\nbasis - lowrank (paired, both at ~50% of rgcn params; positive favours basis)')
        hh = []
        for key in ['indication', 'contraindication', 'micro']:
            a = wide.xs('rgcn_half_basis',   level='arm')[key]
            b = wide.xs('rgcn_half_lowrank', level='arm')[key]
            d = (a - b).dropna()
            hh.append({'metric': key, 'delta': d.mean(), 'CI95': _ci95(d),
                       'n_paired': len(d), 'basis wins': int((d > 0).sum())})
        display(pd.DataFrame(hh).style.format('{:.4f}', subset=['delta', 'CI95'], na_rep='-')
                                .hide(axis='index'))

---
## 8 - Reading the result

Together with `ablation_encoder_swap`, this completes the capacity axis at a fixed graph and fixed
metric learning. The claim it supports or refutes:

> The -0.227 indication AUPRC cost of replacing TxGNN's R-GCN with a shared-weight GNN is a **capacity**
> effect, not a message-passing one. Halving the encoder's parameters while keeping full relational
> message passing costs only X.

Fill `X` in from section 7.1. The three outcomes and what each licenses:

1. **Both half arms within ~0.02 of `rgcn`.** The curve is saturating: TxGNN is heavily
   over-parameterised in its encoder and the published architecture buys little with its last 500k
   parameters. Report the half-arm result as the headline and `light_gnn` as the cliff.
2. **Both roughly halfway to `light_gnn`.** Encoder size buys performance smoothly. The honest wording
   becomes "encoder capacity, not graph breadth, is what the model is spending its performance on" -
   still the inverse of the knowledge-graph-foundation-model framing, but with no over-provisioning
   claim attached.
3. **The two disagree.** The binding constraint is identified, which is a better result than either: if
   `basis` holds and `lowrank` does not, relations need full-rank transforms but not independent ones -
   and then `light_gnn` fails only because 1 basis is too few, not because weights are shared. The
   head-to-head table at the end of section 7 tests this directly.

### What this still does not address

Unchanged from `ablation_encoder_swap`, and none of it is fixed by this notebook:

1. **Frozen node features.** `initialize_node_embedding` (`utils.py:1127`) makes node features frozen
   xavier-random (`requires_grad=False`), so the encoder is the only route by which node identity
   reaches the decoder. That is why `mlp` collapses, and it is a confound in *every* encoder arm here.
   The missing control is a learnable `nn.Embedding` table feeding the same encoder.
2. **One split protocol and the lenient metric.** `complex_disease`, edge-level AUPRC, one sampled
   negative per positive, chance = 0.500.
3. **Pretraining is held fixed** at one epoch in every arm. A smaller encoder may be pretraining-limited
   rather than capacity-limited; a `pretrain n_epoch=0` arm would separate those.
4. **The rank/basis values are not tuned.** `RANK` and `N_BASES` were picked to hit 50% of the parameter
   count, not to maximise anything. A sweep (`rank` in 5, 10, 25, 50) would locate the cliff, at roughly
   26 min per run per seed.